In [0]:
from pyspark.sql import functions as F
from datetime import datetime

ultimo_df = spark.table('vg_sales.01_bronze.stg_vg_sales_cleaned')
#display(ultimo_df)

df_valid = ultimo_df.withColumn("is_valid", (
F.col('console_hash') .isNotNull() &
F.col('publisher_hash') .isNotNull() &
F.col('genre_hash') .isNotNull() & 
F.col('id_lote').isNotNull() &
F.col('title').isNotNull() &
F.col('console').isNotNull() &
F.col('genre').isNotNull() &
F.col('publisher').isNotNull() &
F.col('developer').isNotNull() &
F.col('total_sales').isNotNull() &
F.col('release_date').isNotNull() &
F.col('source_file').isNotNull() &
F.col('ingested_at').isNotNull()
))

#DataFrame Aprovados ⇒ (Silver) | DataFrame Reprovados ⇒ (Quarentena)
df_aprovado = df_valid.filter(F.col("is_valid") == True).drop("is_valid")
df_reprovado = df_valid.filter(F.col("is_valid") == False)

#---------------------------------------------------------------------------------------
#Regra para Reprovados ⇒ (Quarentena)
if df_reprovado.count() > 0:
    df_quarentena = df_reprovado.withColumn("Motivo_Erro", 
        F.concat_ws(" | ", 
            F.when(F.col("title").isNull(), "Titulo_Ausente").otherwise(F.lit(None)),
            F.when(F.col("release_date").isNull(), "Data_Invalida_Nula").otherwise(F.lit(None)),
            F.when(F.col("total_sales").isNull(), "Venda_Nao_Numero").otherwise(F.lit(None)),
            F.when(F.col("total_sales") < 0, "Venda_Negativa").otherwise(F.lit(None))
        )
    ).withColumn("Data_Erro", F.current_timestamp())

    # Salva na Quarentena
    df_quarentena.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("vg_sales.01_bronze.quarentena_vg_sales")
    print(f"⚠️ {df_reprovado.count()} registros movidos para a quarentena.")

#---------------------------------------------------------------------------------------
#Regra para Aprovados ⇒ (Go to Silver)

if df_aprovado.count() > 0:
    print(f"✅ {df_aprovado.count()} registros aprovados. Iniciando Merge total na Silver...")

    # 4.1. Adicionamos apenas as colunas de controle que a Silver exige
    # (Release Year para partição e Updated At para auditoria)
    df_to_upsert = df_aprovado.withColumn("release_year", F.year(F.col("release_date"))) \
                             .withColumn("updated_at", F.current_timestamp())

    # 4.2. Criação da View Temporária
    df_to_upsert.createOrReplaceTempView("vw_lote_aprovado")

    # 4.3. MERGE INTEGRAL
    # O "UPDATE SET *" e "INSERT *" garantem que TODAS as colunas do df_aprovado
    # entrem na Silver, sem precisar listar uma por uma.
    spark.sql("""
        MERGE INTO vg_sales.02_silver.fact_vg_sales_main AS target
        USING vw_lote_aprovado AS source
        ON target.id_lote = source.id_lote
        WHEN MATCHED THEN 
            UPDATE SET *
        WHEN NOT MATCHED THEN 
            INSERT *
    """)

    status_job = "Sucesso"
    print(f"🚀 Merge concluído!")

else:
    status_job = "Aviso"
    print("info: Nenhum dado válido para carregar. Verifique a tabela de quarentena.")


#---------------------------------------------------------------------------------------
#Log Final ⇒ (logs_vgsales_bronze_to_silver)

# 1. Captura as métricas reais do processo
# Usamos variáveis que calculamos durante o try/except
rows_silver = df_aprovado.count() if 'df_aprovado' in locals() else 0
rows_quarentena = df_reprovado.count() if 'df_reprovado' in locals() else 0
total_silver_final = spark.table("vg_sales.02_silver.fact_vg_sales_main").count()

# 2. Extrai info do lote (usando o ultimo_df que você definiu no início)
info_lote = ultimo_df.select(
    F.max("ingested_at").alias("batch_ingested_at"),
    F.first("source_file").alias("file_name")
).first()

# 3. Monta a estrutura do Log
log_data = [(
    datetime.now(),                    # audit_date (agora)
    info_lote["batch_ingested_at"],    # batch_ingested_at (data do lote)
    info_lote["file_name"],            # nome do arquivo
    rows_silver,                       # registros aprovados
    rows_quarentena,                   # registros reprovados
    total_silver_final,                # tamanho total da silver pos-merge
    status_job                         # status definido no seu if/else
)]

schema_log = """
    audit_date timestamp, 
    batch_ingested_at timestamp, 
    file_name string, 
    rows_silver long, 
    rows_quarentena long, 
    total_rows_silver long, 
    status string
"""

# 4. Grava na Bronze (Histórico de Logs)
df_log_final = spark.createDataFrame(log_data, schema=schema_log)

df_log_final.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("vg_sales.01_bronze.logs_vgsales_bronze_to_silver")

print(f"🏁 Processo finalizado com status: {status_job}")
print(f"📊 Silver: +{rows_silver} | Quarentena: +{rows_quarentena}")
